# Evaluating Trained FM Models' Performance

In [ ]:
import os
import sys

from corner import corner

sys.path.append('..')

from src.simulator import Model, BurstSimulator
from src.flow_matching.distributions import UniformPrior, CompositePrior, Posterior
from src.flow_matching.probability_path import GuidedLinearProbabilityPath
from src.flow_matching.integration import EulerODESolver
from src.flow_matching.models import MLPGuidedVectorField, FRBLightCurveCNN, LightCurveThinner, fourier_embedding, LightCurveMLP
from src.flow_matching.transformer import TransformerGuidedField
from src.helpers import record_every, plot_posterior_samples, gen_parameter_labels

import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
import torch
import yaml

from src.flow_matching.plotting import plot_loss, plot_snapshots
from src.flow_matching.helpers import choose_device, build_mlp, find_run_dir

device = choose_device()

In [ ]:
MODELS = {
    "MLPGuidedVectorField": MLPGuidedVectorField,
    "TransformerGuidedField":TransformerGuidedField
}

TIME_ENCODERS = {
    "LightCurveThinner": LightCurveThinner,
    "FRBLightCurveCNN": FRBLightCurveCNN,
    "LightCurveMLP":LightCurveMLP
}

TAU_ENCODERS = {
    True : fourier_embedding,
    False : None 
}

THETA_ENCODERS = {
    # True : lambda theta_dim : build_mlp([2] + [2 * 4, 2 * 8] + [theta_dim]),
    True : lambda theta_dim : build_mlp([3] + [3 * 8, 3 * 32] + [theta_dim]),
    False : None
}

PATHS = {
    "GuidedLinearProbabilityPath": GuidedLinearProbabilityPath,
}

DISTRIBUTIONS = {
    "Posterior":     Posterior,
    "UniformPrior":  UniformPrior,
    "CompositePrior":CompositePrior,
    "NewPosterior":  Posterior
}

In [ ]:
# fill in desired job_id or directory name 
job_id = "13087631" #False
run_dir = None
save_dir = "../checkpoints/"

In [ ]:
# loading the model (via run_id, or path)
run_dir = find_run_dir(job_id, save_dir) if job_id else os.path.join(save_dir, run_dir)

checkpoint_path = os.path.join(run_dir, 'training_checkpoint.pth')
config_path     = os.path.join(run_dir, 'config.yaml')

In [ ]:
# create empty model from config
def empty_model_from_config(path):
    with open(path, 'r') as f:
        config = yaml.safe_load(f)
    
    model_name = config["model"]["name"]
    model_class = MODELS[model_name]
    
    kwargs = config["model"]["init_params"]

    time_encoder = False if not config["time_seq_encoder"] else config["time_seq_encoder"]["name"] 

    if time_encoder:
        t_encoder_kwargs = config["time_seq_encoder"]["init_params"]


    kwargs['time_seq_encoder']  = None if not time_encoder else TIME_ENCODERS[time_encoder](**t_encoder_kwargs)
    kwargs['tau_encoder']   = TAU_ENCODERS[config["tau_encoder"]]
    kwargs['theta_encoder'] = THETA_ENCODERS[config["theta_encoder"]]

    return model_class(**kwargs)

vector_field = empty_model_from_config(config_path)

In [ ]:
# load trained model 
print(checkpoint_path)
checkpoint = torch.load(checkpoint_path, weights_only=False)

# Load 'normal' or EMA version 
ema = False
if ema:
    EMA_checkpoint_path = os.path.join(run_dir, "EMA_checkpoint.pth")
    state_dict = torch.load(EMA_checkpoint_path, weights_only=False)
    vector_field.load_state_dict(state_dict) 
else:
    vector_field.load_state_dict(checkpoint["model_state_dict"])

vector_field.eval()
vector_field.to(device)

losses = checkpoint["losses"]

In [ ]:
# loading the probability path
def prob_path_from_config(path):

    with open(path, 'r') as f:
        config = yaml.safe_load(f)
    
    path_name = config["path"]["name"]
    path_class = PATHS[path_name]
    
    inf_params = config["path"]["p_data"]["init_params"]['inf_params']
    p_simple_name = config["path"]["p_simple"]["name"]
    p_simple_cls = DISTRIBUTIONS[p_simple_name]
    kwargs = config["path"]["p_simple"]["init_params"]
    
    if p_simple_name == "CompositePrior":
        prior_dict = {}

        for param, prior in zip(inf_params, kwargs):
            prior_cls = DISTRIBUTIONS[prior['name']]
            prior_dict[param] = prior_cls(**prior['init_params'])

        p_simple = p_simple_cls(prior_dict)

    else:
        p_simple = p_simple_cls(**kwargs)
    
    p_data_name   = config["path"]["p_data"]["name"]
    p_data_cls = DISTRIBUTIONS[p_data_name]

    kwargs = config["path"]["p_data"]["init_params"]
    kwargs.pop('prior')
    
    p_data = p_data_cls(prior=p_simple, **kwargs)
    path = path_class(p_simple, p_data)
    
    return path

path = prob_path_from_config(config_path)

# Loss

In [ ]:
plot_loss(losses, window_size=10)

# Example posterior

In [ ]:
N = 1
time = np.linspace(0, 1.0, 1000)
amp  = 100.0
rise = 0.03
skew = 5
ybkg = 5.0
true_t0 = [0.5] # NOTE: this has to obey the prior! (i.e. t0_1 < t0_2 < ... < t0_n)

inf_params = path.p_data.inf_params

# generate simulated data for one burst
burstparams = {
    't0'   : true_t0, 
    'amp'  : [amp for _ in range(N)],
    'rise' : [rise for _ in range(N)],
    'skew' : [skew for _ in range(N)]
}

modelparams = {
    "time": time,
    "ncomp": N,
    "burstparams":burstparams,
    "ybkg":ybkg
}

# generate one instance of simulated data to guide prior samples with
model = Model(**modelparams)
simulator = BurstSimulator(model)
x_counts = simulator.simulate_burst() 
simulator.plot_burst()

In [ ]:
num_samples = 25000  # number of prior samples to transform 
num_marginals = 5   # number of snapshots

# TODO: maybe do this in batches
# use same data point for conditioning all prior samples
simulations = torch.tensor(x_counts, device=device, dtype=torch.float).repeat(num_samples, 1)

# initialize ODE solver
solver = EulerODESolver(vector_field)
nts = 200
ts = torch.linspace(0, 1, nts).to(device)
# ts_2 = torch.linspace(0.90, 1, nts - 100).to(device)
# ts = torch.cat([ts, ts_2])

# simulate ODE starting from x0
x0 = path.p_simple.sample(num_samples).to(device)

xts = solver.solve_with_trajectory(x0, ts.view(1, nts, 1).expand(num_samples, nts, 1), y=simulations)

# only save num_marginals snapshots 
record_every_idxs = record_every(nts, nts // (num_marginals - 1))
xts_snapshots = xts[:, record_every_idxs, :]

# plot snapshots of marginal path
final_snapshot = plot_snapshots(xts_snapshots, ts, record_every_idxs, num_marginals, vector_field.inf_params, vector_field.dim // len(vector_field.inf_params))

In [ ]:
import matplotlib.pyplot as plt

final_snapshot = xts[:, -1, :].cpu()
penultimate_snapshot = xts[:, -2, :].cpu()
plt.figure(figsize=(13, 5))

plt.subplot(121)
plt.title(f'second to last snapshot (t={ts[-2]:.3f})')
plt.hist2d(penultimate_snapshot[:,0], penultimate_snapshot[:,1], bins=75, cmin=1)
plt.colorbar()
plt.xlim(0.290, 0.315)
plt.ylim(0.68, 0.71)
plt.axis('equal')

plt.subplot(122)
plt.title(f'last snapshot (t={ts[-1]:.3f})')
plt.hist2d(final_snapshot[:,0], final_snapshot[:,1], bins=75, cmin=1)
plt.colorbar()
plt.xlim(0.290, 0.315)
plt.ylim(0.68, 0.71)
plt.axis('equal')
plt.show()

In [ ]:
# plot loss over time
samples = 1000
ts_ = torch.linspace(0.0, 1.0, samples).to(device).view(-1, 1)
x0_ = path.p_simple.sample(samples).to(device)
simulations_ = torch.tensor(x_counts, device=device, dtype=torch.float).repeat(samples, 1)
target_ = torch.tensor([[0.3, 0.7]], device=device).repeat((samples, 1))

print(x0_.shape, ts_.shape, simulations_.shape)
differences_ = (
            vector_field(x0_, ts_, simulations_)
            - path.conditional_vector_field(x0_, target_)
            ) # shape batch_size, ndim

losses_ = differences_.norm(dim=1) 
plt.title("MSE loss norm as a func of tau")
plt.plot(ts_.cpu().detach().numpy(), losses_.cpu().detach().numpy(), '-', linewidth=1)
plt.xlabel("time $\\tau$")
plt.ylabel('loss norm')
# plt.yscale('log')

plt.show()

plt.title("vector field norm sqrd as a func of tau")
plt.plot(ts_.cpu().detach().numpy(), torch.sum(vector_field(x0_, ts_, simulations_) ** 2, dim = 1).cpu().detach().numpy(),
label="learned")
plt.ylabel("$||u_t(x)||^2$")
plt.xlabel("time $\\tau$")

plt.plot(ts_.cpu().detach().numpy(), torch.sum(
    path.conditional_vector_field(x0_, target_) ** 2,
    dim = 1 # sum column wise
).cpu().detach().numpy(), label="truth (conditional)")
plt.legend()
plt.show()
# plt.yscale('log')

plt.title('learned u_t norm')
plt.plot(ts_.cpu().detach().numpy(),
    vector_field(x0_, ts_, simulations_).norm(dim=1).cpu().detach().numpy())
plt.ylabel("$|u_t(x)|$")
plt.xlabel("time $\\tau$")
plt.show()

In [ ]:
nts_test = 6
x0_test = path.p_simple.sample(nts_test - 1).to(device)
x0_test = torch.cat((x0_test, torch.tensor([[100, 100]], device=device)))

In [ ]:
ts_test = torch.linspace(0.5, 1.0, nts_test).to(device)
simulations_test = torch.tensor(x_counts, device=device, dtype=torch.float).repeat(nts_test, 1)
ts_test = ts_test.view(-1, 1)

print(x0_test, ts_test)
target = torch.tensor([[100, 100]], device=device).repeat((nts_test, 1))
print("truth:", path.conditional_vector_field(x0_test, target))
print("learned:", vector_field(x0_test, ts_test, simulations_test))
differences = (
            vector_field(x0_test, ts_test, simulations_test)
            - path.conditional_vector_field(x0_test, target)
            ) # shape batch_size, ndim

losses = torch.sum(
    differences ** 2,
    dim = 1 # sum column wise
)
total_loss = torch.sum(losses)

average =  total_loss / nts_test
print(average)
if torch.isnan(average):
    print("lol")

In [ ]:
animate = False

if animate:
    fig, ax = plt.subplots(figsize=(7,7))

    # histogram bin range
    bins = 100
    x_edges = np.linspace(0.2, 0.4, bins + 1)
    y_edges = np.linspace(0.6, 0.8, bins + 1)

    # initial histogram
    xts = xts.cpu()
    hist, xedges, yedges, img = ax.hist2d(xts[:, 0, 0], xts[:, 0, 1], bins=[x_edges, y_edges], cmin=1)
    # plt.colorbar(img, ax=ax)
    # ax.set_title('index = 0')

    def update(frame):
        ax.clear()  
        hist, xedges, yedges, img = ax.hist2d(xts[:, frame, 0], xts[:, frame, 1],
                                            bins=100, 
                                            cmin=1)
        ax.set_title(f'timestep = {frame}')
        ax.set_xlim(x_edges[0], x_edges[-1])
        ax.set_ylim(y_edges[0], y_edges[-1])
        ax.plot(true_t0[0], true_t0[1], 'ro', markersize=4)
        
        # plt.colorbar(img, ax=ax)
        return img,

    timesteps = np.arange(900, 1000, 2)
    ani = animation.FuncAnimation(fig, update, frames=timesteps,
                                interval=200, blit=False)
    ani.save('animation2.gif')
    plt.show()

In [ ]:
fig, axes = plt.subplots(1,1, figsize=(12, 12))
ax = axes
ax.set_title('trajectories of ODE', fontsize=20)
# ax.scatter(x0[:,0].cpu(), x0[:,1].cpu(), marker='*', color='red', s=200, label='z',zorder=20) # Plot z

x_bounds=(0,1)
y_bounds=(0,1)

# plot N random trajctories
import random
N_traj = 25
traj = [random.choice(range(num_samples)) for i in range(N_traj)]
print(x0.shape)
ax.scatter(x0[traj,0].detach().cpu(), x0[traj,1].detach().cpu(), marker='*', color='red', s=200, label='$\\theta_0$',zorder=20) # Plot z

for traj_idx in traj:
    ax.plot(xts[traj_idx,:,0].detach().cpu(), xts[traj_idx,:,1].detach().cpu(), alpha=0.5, color='black')
ax.legend(loc='upper right', markerscale=1)
plt.show()

# corner plot

In [ ]:
var_names = gen_parameter_labels(inf_params, N)
true_values = np.array([simulator.get_true(key) for key in inf_params]).flatten()
fig = corner(final_snapshot.cpu().numpy(), labels=var_names, truths=true_values, range=[0.99 for _ in range(N * len(inf_params))])

In [ ]:
plot_posterior_samples(100, x_counts, final_snapshot.cpu().numpy(), vector_field.inf_params, modelparams)
true_flux = model.get_flux()
plt.plot(np.linspace(0, 1, len(true_flux)), true_flux, 'r--', linewidth=1, label="ground-truth")
plt.legend()

# PP-plot

In [ ]:
# cell for PP-plot